<a href="https://colab.research.google.com/github/Don4141/ProductPriceAI/blob/main/Modal_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Deploy multiple models to production including proprietary LLMs on Modal, RAG workflow with Frontier models, and Neural Network Models.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
os.chdir("/content")
#Clone repo after a new Colab runtime as the previous will be erased
!git clone https://github.com/Don4141/ProductPriceAI.git

#Set repository as working directory
os.chdir("/content/ProductPriceAI")

#Verify
print("Repo exist", os.path.exists("/content/ProductPriceAI"))
print("Current working directory", os.getcwd())

Cloning into 'ProductPriceAI'...
remote: Enumerating objects: 298, done.
remote: Counting objects: 100% (298/298), done.
remote: Compressing objects: 100% (244/244), done.
remote: Total 298 (delta 151), reused 147 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (298/298), 17.98 MiB | 15.25 MiB/s, done.
Resolving deltas: 100% (151/151), done.
Repo exist True
Current working directory /content/ProductPriceAI


In [3]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1 transformers==4.57.6

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [4]:
!pip uninstall -y gradio gradio-client diffusers

Found existing installation: gradio 6.31.0
Uninstalling gradio-6.31.0:
  Successfully uninstalled gradio-6.31.0
Found existing installation: gradio_client 2.7.2
Uninstalling gradio_client-2.7.2:
  Successfully uninstalled gradio_client-2.7.2
Found existing installation: diffusers 0.42.1
Uninstalling diffusers-0.42.1:
  Successfully uninstalled diffusers-0.42.1


In [5]:
#Manage imports
import re
import math
import numpy as np
import torch
import wandb
import transformers
import trl
import peft
import bitsandbytes
import huggingface_hub
from tqdm import tqdm
from pricer.items import Item
from pricer.evaluator import evaluate, Tester
from tqdm.notebook import tqdm
from IPython.display import display, Image
from pathlib import Path
from transformers import AutoTokenizer
from google.colab import userdata
from huggingface_hub import login
import transformers
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
import matplotlib.pyplot as plt

print("All LLM training libraries imported successfully!")

All LLM training libraries imported successfully!


In [6]:
#Load API credentials
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY", "GITHUB_TOKEN",
            "HF_TOKEN", "MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET"]:
  os.environ[key] = userdata.get(key)

In [7]:
#Verify if API keys are loaded
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY", "GITHUB_TOKEN",
            "HF_TOKEN", "MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET"]:
    print(f"{key}: {'loaded' if os.getenv(key) else 'Missing'}")

OPENAI_API_KEY: loaded
ANTHROPIC_API_KEY: loaded
GROQ_API_KEY: loaded
GITHUB_TOKEN: loaded
HF_TOKEN: loaded
MODAL_TOKEN_ID: loaded
MODAL_TOKEN_SECRET: loaded


In [8]:
#Install modal and ensure modal credentials are successfully loaded
!pip install -q modal

print("Modal Token ID:", bool(os.getenv("MODAL_TOKEN_ID")))
print("Modal Token Secret:", bool(os.getenv("MODAL_TOKEN_SECRET")))

Modal Token ID: True
Modal Token Secret: True


In [9]:
#Verify authentication with Modal
!modal secret list

                                    Secrets                                     
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ Name               ┃ Created at           ┃ Created by ┃ Last used at        ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ huggingface-secret │ 2026-11-19 10:39 UTC │ don4141    │ 2026-11-21 12:21    │
│                    │                      │            │ UTC                 │
└────────────────────┴──────────────────────┴────────────┴─────────────────────┘


In [10]:
%pip install -q -r requirements.txt

In [11]:
import locale
import modal
from agents.preprocessor import Preprocessor

In [12]:
import importlib
import agents.preprocessor

importlib.reload(agents.preprocessor)

Preprocessor = agents.preprocessor.Preprocessor

preprocessor = Preprocessor()

print("Model:", preprocessor.model_name)

Model: groq/openai/gpt-oss-20b


In [13]:
preprocessor = Preprocessor()
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
print(text)

Title: Quadcast HyperX Condenser Microphone  
Category: Audio Equipment  
Brand: HyperX  
Description: A USB-C condenser mic delivering crystal‑clear sound for streaming, podcasting, and gaming.  
Details: Features dual‑polar pattern switching, built‑in headphone monitoring, and on‑board mute control.


In [83]:
with modal.enable_output():
    with app.run():
        result = price.remote(text)
print(result)

✓ Initialized. View run at 
https://modal.com/apps/don4141/main/ap-yFo5xkY0zeUnzafMaCfe7d
✓ Created objects.
├── ✓ Uploaded PythonPackage:other
└── ✓ Created Function price.

Fetching 2 files: 100%|██████████| 2/2 [00:21<00:00, 10.79s/it]

Loading weights: 100%|██████████| 254/254 [00:02<00:00, 118.53it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Stopping app - local entrypoint completed.
✓ App completed. View run at 
https://modal.com/apps/don4141/main/ap-yFo5xkY0zeUnzafMaCfe7d
110.0


In [89]:
#Deploy model
!modal deploy /content/ProductPriceAI/other/pricer_service2.py

⠦ Creating objects...
⠏ Creating objects...
└── ⠸ Creating mount /content/ProductPriceAI/other/pricer_service2.py: Uploaded 
⠹ Creating objects...
└── ⠦ Creating mount /content/ProductPriceAI/other/pricer_service2.py: Uploaded 
⠴ Creating objects...
└── ⠏ Creating mount /content/ProductPriceAI/other/pricer_service2.py: 
⠇ Creating objects...
├── ✓ Uploaded /content/ProductPriceAI/other/pricer_service2.py
⠙ Creating objects...
├── ✓ Uploaded /content/ProductPriceAI/other/pricer_service2.py
⠼ Creating objects...
├── ✓ Uploaded /content/ProductPriceAI/other/pricer_service2.py
└── ✓ Created Function Pricer.*.
✓ Created objects.
├── ✓ Uploaded /content/ProductPriceAI/other/pricer_service2.py
└── ✓ Created Function Pricer.*.
✓ App deployed in 3.037s! 🎉

View Deployment: ]8;id=426966;https://modal.com/apps/don4141/main/deployed/pricer-service\https://modal.com/apps/don4141/main/deployed/pricer-service]8;;\


In [14]:
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()
reply = pricer.price.remote(text)
print(reply)

60.0


In [99]:
reply = pricer.price.remote(text)
print(reply)

60.0


In [15]:
from agents.specialist_agent import SpecialistAgent

In [16]:
agent = SpecialistAgent()

In [17]:
agent.price("iPhone 13 Pro Max")

655.0

RAG (Retrieval Augmented Generation) based on a dataset of 800,000 scraped Amazon products.

In [ ]:
#Imports
import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
from tqdm.notebook import tqdm
from agents.evaluator import evaluate
from agents.items import Item

In [ ]:
# environment
load_dotenv(override=True)
DB = "products_vectorstore"

In [ ]:
# Log in to HuggingFace
# If you don't have a HuggingFace account, you can set one up for free at www.huggingface.co
# And then add the HF_TOKEN to your .env file as explained in the project README

hf_token = os.environ['HF_TOKEN']
login(token=hf_token, add_to_git_credential=False)

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Now create a Chroma Datastore

In [ ]:
client = chromadb.PersistentClient(path=DB)

Introducing the SentenceTransformer Encoding LLM
The all-MiniLM is a very useful model from HuggingFace that maps sentences & paragraphs to 384 dimensional vectors and is ideal for tasks like semantic search.

https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

It can run pretty quickly locally.

As an alternative, OpenAI provides a closed-source Embeddings model. Benefits compared to OpenAI embeddings:

It's free and fast!
We can run it locally, so the data never leaves our box - might be useful if you're building a personal RAG

In [ ]:
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
# Pass in a list of texts, get back a numpy array of vectors
vector = encoder.encode(["A proficient AI engineer who has almost reached the finale of AI Engineering Core Track!"])[0]
print(vector.shape)
vector

# Pass in a list of texts, get back a numpy array of vectors

vector = encoder.encode(["A proficient AI engineer who has almost reached the finale of AI Engineering Core Track!"])[0]
print(vector.shape)
vector

In [ ]:
#Create a vector space for all 800,000 description of the products in the train dataset
# Check if the collection exists; if not, create it

collection_name = "products"
existing_collection_names = [collection.name for collection in client.list_collections()]

if collection_name not in existing_collection_names:
    collection = client.create_collection(collection_name)
    for i in tqdm(range(0, len(train), 1000)):
        documents = [item.summary for item in train[i: i+1000]]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+1000]]
        ids = [f"doc_{j}" for j in range(i, i+1000)]
        ids = ids[:len(documents)]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)

collection = client.get_or_create_collection(collection_name)

Let's visualize the vectorized data

In [ ]:
# It is very fun turning this up to 800_000 and seeing the full dataset visualized,
# but it almost crashes my box every time so do that at your own risk!! 10_000 is safe!

#Subset and visualize 10,000 of the vectors representing 10,000 of the train dataset stored in chroma DB
MAXIMUM_DATAPOINTS = 10_000

In [ ]:
#Specifying the categories of the products associated with the transformed vectors stored in Chroma DB
CATEGORIES = ['Appliances', 'Automotive', 'Cell_Phones_and_Accessories', 'Electronics','Musical_Instruments', 'Office_Products', 'Tools_and_Home_Improvement', 'Toys_and_Games']
COLORS = ['cyan', 'blue', 'brown', 'orange', 'yellow', 'green' , 'purple', 'red']

In [ ]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'], limit=MAXIMUM_DATAPOINTS)
vectors = np.array(result['embeddings'])
documents = result['documents']
categories = [metadata['category'] for metadata in result['metadatas']]
colors = [COLORS[CATEGORIES.index(c)] for c in categories]

In [ ]:
# TSNE stands for t-distributed Stochastic Neighbor Embedding - it's a common technique for reducing dimensionality of data
#Use T-distributed Stochastic Neighbour Embedding to project down the 384 dimensions that the encoder came down with down to two dimensions, and display that in a plot
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=4, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vectorstore Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# Let's try 3D!
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=2, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
#Create a function that takes an item discription and transform it into a vector using encoder
def vector(item):
    return encoder.encode(item.summary)

In [ ]:
#Define a function that when given an item, it will return similar items and their prices
def find_similars(item):
    vec = vector(item)
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=5)
    documents = results['documents'][0][:]
    prices = [m['price'] for m in results['metadatas'][0][:]]
    return documents, prices

In [ ]:
find_similars(test[0])

In [ ]:
# We need to give some context to GPT-5.1 by selecting 5 products with similar descriptions

#Create a function such that when given a context which is the description of a product,
#it will search through the vectorstore created and return related products and their associated prices.
def make_context(similars, prices):
    message = "For context, here are some other items that might be similar to the item you need to estimate.\n\n"
    for similar, price in zip(similars, prices):
        message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
    return message

In [ ]:
documents, prices = find_similars(test[0])
print(make_context(documents, prices))

In [ ]:
#Turn above function into an open AI style messages
def messages_for(item, similars, prices):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}\n\n"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]

In [ ]:
documents, prices = find_similars(test[0])
print(messages_for(test[0], documents, prices)[0]['content'])

In [ ]:
# The function for gpt-5.1
#Create a function that calls GPT5.1 in RAG mode. This function will take the description of an item, find related items from the RAG created
#The call GPT 5.1 and pass in the item description and the retrieved related items as extra context
def gpt_5__1_rag(item):
    documents, prices = find_similars(item)
    response = completion(model="gpt-5.1", messages=messages_for(item, documents, prices), reasoning_effort="none", seed=42)
    return response.choices[0].message.content

In [ ]:
# How much does our favorite distortion pedal cost?
test[0].price

In [ ]:
#Thi run will first look up in the Chroma database, find similar items and then make a call to GPT 5.1 in the cloud to come up with an estimate of the product
gpt_5__1_rag(test[0])

In [ ]:
#Apply that to the entire test set
evaluate(gpt_5__1_rag, test)

In [ ]:
#
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()

In [ ]:
#Calls modal in the cloud and returns a summary of an item from our deployed model on modal
def specialist(item):
    return pricer.price.remote(item.summary)

In [ ]:
#Function to strip out a number when the model returns a text
def get_price(reply):
    reply = reply.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
    return float(match.group()) if match else 0

Download the Neural Network weights from week 6

In [ ]:
from agents.deep_neural_network import DeepNeuralNetworkInference

runner = DeepNeuralNetworkInference()
runner.setup()
runner.load("deep_neural_network.pth")
#Use the ANN model trained in week 6 to also predict the price of a the given item
def deep_neural_network(item):
    return runner.inference(item.summary)

In [ ]:
#Create an emsemble function that will take the frontier RAG (GPT 5.1 and RAG), the specialist model and the ANN model
#This combines 80% of the output from the frontier RAG, 10% of the specialist model and 10% of the ANN and returns an average price
def ensemble(item):
    price1 = get_price(gpt_5__1_rag(item))
    price2 = specialist(item)
    price3 = deep_neural_network(item)
    return price1 * 0.8 + price2 * 0.1 + price3 * 0.1

In [ ]:
#Apply the above function on the entire test dataset
evaluate(ensemble, test)

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
#Put the notebook into production ready python module
from agents.frontier_agent import FrontierAgent

agent = FrontierAgent(collection)
agent.price("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")

In [ ]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

In [ ]:
from agents.neural_network_agent import NeuralNetworkAgent
agent = NeuralNetworkAgent()

In [ ]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

In [ ]:
from agents.ensemble_agent import EnsembleAgent
agent = EnsembleAgent(collection)

In [ ]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")